## Melatih Model _(training)_
<p>Model akan "belajar" dari data training dengan menyesuaikan parameternya agar dapat memetakan input (fitur) ke output (target) dengan baik. Selama pelatihan, model akan menyesuaikan bobot atau koefisiennya untuk meminimalkan kesalahan antara prediksi dan nilai sebenarnya dalam training set.</p>
<p>Ada dua hal yang perlu Anda perhatikan pada tahapan ini, yaitu fitur dan target. Fitur adalah data input yang digunakan untuk melatih model, sedangkan target adalah data output yang menjadi referensi model untuk belajar.</p>
<p>Perlu Anda catat, pada latihan ini kita tidak akan melakukan hyperparameter tuning sehingga algoritma yang digunakan akan menghasilkan output berdasarkan konfigurasi dasarnya. Sebagai pemanasan, mari kita latih data yang sudah kita miliki dengan tiga algoritma yang berbeda.</p>

##### Handle Missing Values, Label Encoding, Cek Missing Value, Standardisasi, lalu Data Division
Load Dataset dulu

In [62]:
import pandas as pd

# Load dataset:
df = pd.read_csv(r"dataset/train.csv")

Missing Values

In [79]:
# Contoh mengisi nilai yang hilang dengan median untuk kolom numerik
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())
    # print(df[col])

categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

Label Encoding

In [80]:
from sklearn.preprocessing import LabelEncoder

# Label encoding untuk kolom kategorical aja
df_lencoder = df.copy()
for col in categorical_cols:
    df_lencoder[col] = LabelEncoder().fit_transform(df[col])

Cek Missing Values

In [81]:
# Cek missing value
print(f"Missing Values: {df_lencoder.isnull().sum().sum()}")

Missing Values: 0


Pisahkan X dan Y (Note: ini bukan data division.)

In [82]:
# Memisahkan fitur (X) dan target (y)
X = df_lencoder.drop(columns=['SalePrice'])
y = df_lencoder['SalePrice']

print(f"X Shape: {X.shape}")
print(f"y Shape: {y.shape}")

X Shape: (1460, 80)
y Shape: (1460,)


Split Data (sebelum standardisasi)

In [83]:
from sklearn.model_selection import train_test_split
# Split dulu
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Standardisasi

In [84]:
from sklearn.preprocessing import StandardScaler

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
scaler = StandardScaler()
# X[numeric_features] = scaler.fit_transform(X[numeric_features])
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

##### ...Lanjut dengan Pembangunan Model

In [85]:
# Melatih model 1 dengan algoritma Least Angle Regression
from sklearn import linear_model
lars = linear_model.Lars().fit(x_train, y_train) # hapus n_nonzero_coefs=1
 
# Melatih model 2 dengan algoritma Linear Regression
from sklearn.linear_model import LinearRegression
LR = LinearRegression().fit(x_train, y_train)
 
# Melatih model 3 dengan algoritma Gradient Boosting Regressor
from sklearn.ensemble import GradientBoostingRegressor
GBR = GradientBoostingRegressor(random_state=184)
GBR.fit(x_train, y_train)

print("Semua model berhasil dilatih!")

Semua model berhasil dilatih!


## Evaluasi Model
<p>Model yang telah dilatih perlu melalui tahapan evaluasi berdasarkan validation set untuk melihat seberapa baik ia mampu memprediksi output yang benar dari input yang belum pernah dilihat sebelumnya. Metrik umum untuk evaluasi adalah <i>accuracy, precision, recall, F1-score</i> (untuk klasifikasi), dan <i>Mean Squared Error</i> (MSE) atau <i>R-squared</i> (untuk regresi).</p>
<p>Karena contoh kasus yang sedang kita hadapi merupakan regresi, evaluasi yang akan akan kita gunakan adalah MAE, MSE, dan R2. Untuk melakukan evaluasi ini kita membutuhkan validation set (x_test). Validation test merupakan bagian dari data yang tidak digunakan untuk pelatihan tetapi digunakan untuk mengevaluasi model (<i>unseen data</i>). Mari kita lakukan evaluasi satu per satu.</p>

In [86]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Evaluasi model pada LARS
pred_lars = lars.predict(x_test)
mae_lars = mean_absolute_error(y_test, pred_lars)
mse_lars = mean_squared_error(y_test, pred_lars)
r2_lars = r2_score(y_test, pred_lars)

# Membuat dictionary untuk menyimpan hasil evaluasi
data = {
    'MAE': [mae_lars],
    'MSE': [mse_lars],
    'R2': [r2_lars]
}

# Konversi dictionary menjadi DataFrame
df_results = pd.DataFrame(data, index=['Lars'])
df_results

,MAE,MSE,R2
Lars,35651.347385,2.310330e+09,0.698796


<p>Hasil yang Ideal:</p>

![alt text](image/dos-7a8b538a71b5c9bc201649e8c1c793c720241015131321.jpg)

In [87]:
# Evaluasi pada model Linear Regression
pred_LR = LR.predict(x_test)
mae_LR = mean_absolute_error(y_test, pred_LR)
mse_LR = mean_squared_error(y_test, pred_LR)
r2_LR = r2_score(y_test, pred_LR)
 
# Menambahkan hasil evaluasi LR ke DataFrame
df_results.loc['Linear Regression'] = [mae_LR, mse_LR, r2_LR]
df_results

,MAE,MSE,R2
Lars,35651.347385,2.310330e+09,0.698796
Linear Regression,21598.396868,1.249213e+09,0.837137


Hasil yang Ideal:

![alt text](image/dos-00538340da54326a00102ca6d8cdb6a720241015131321.jpg)

In [91]:
# Evaluasi pada model Gradient Boosting Regressor
pred_GBR = GBR.predict(x_test)
mae_GBR = mean_absolute_error(y_test, pred_GBR)
mse_GBR = mean_squared_error(y_test, pred_GBR)
r2_GBR = r2_score(y_test, pred_GBR)
 
# Menambahkan hasil evaluasi GBR ke DataFrame
df_results.loc['GradientBoostingRegressor'] = [mae_GBR, mse_GBR, r2_GBR]
df_results

,MAE,MSE,R2
Lars,35651.347385,2.310330e+09,0.698796
Linear Regression,21598.396868,1.249213e+09,0.837137
GradientBoostingRegressor,17358.834884,8.963802e+08,0.883137


Hasil yang ideal:

![alt text](image/dos-4d55b9f24df1ac01bd1ddb524732a00020241015131321.jpg)

<p>Sampai di sini, mudah ‘kan? Seperti yang dapat Anda lihat dari beberapa kode di atas memiliki struktur yang sama, yaitu .predict() dan beberapa metriks evaluasi seperti MAE, MSE, dan R2. Mari kita pelajari apa fungsi dari masing-masing function tersebut.</p>
<ul>
    <li style="margin-bottom: 1.5%;"><b>.predict():</b> fungsi <b>.predict()</b> pada scikit-learn digunakan untuk membuat prediksi berdasarkan model yang telah dilatih. Setelah Anda melatih model dengan data pelatihan (menggunakan .fit() pada materi sebelumnya), Anda dapat menggunakan <span style="color:white; background-color:blue; padding: 0.2% 0.5% 0.2% 0.5%; border-radius:5px;">.predict()</span> untuk menghasilkan nilai prediksi pada data baru atau data testing.</li>
    <li style="margin-bottom: 1.5%;"><b>mean_absolute_error:</b> MAE mengukur rata-rata dari kesalahan absolut antara nilai prediksi dan nilai aktual. Ini adalah ukuran yang intuitif karena langsung menghitung seberapa jauh prediksi dari nilai sebenarnya tanpa memperhitungkan arah (positif atau negatif).</li>
    <li style="margin-bottom: 1.5%;"><b>mean_squared_error:</b> MSE mengukur rata-rata dari kuadrat kesalahan antara nilai prediksi dan nilai aktual. Karena kesalahan dikuadratkan, MSE memberikan penalti (nilai error) yang lebih besar untuk kesalahan yang lebih besar, membuatnya lebih sensitif terhadap outlier.</li>
    <li style="margin-bottom: 1.5%;"><b>r2_score:</b> R² adalah metrik statistik yang menunjukkan seberapa baik nilai prediksi mendekati nilai aktual. R² mengukur proporsi varians dari target yang dapat dijelaskan oleh fitur dalam model.</li>
</ul>
<p>Nah, berdasarkan hasil evaluasi terhadap unseen data di atas, tentunya Anda sudah lebih yakin terhadap performa model yang telah dibangun. Lalu, manakah model yang Anda pilih berdasarkan latihan di atas? Jika memilih GradientBoostingRegressor berarti Anda sudah memahami arti dari masing-masing nilai evaluasi di atas. Selanjutnya mari kita selesaikan latihan machine learning workflow yang sangat panjang ini dengan menyimpan model agar dapat digunakan pada lingkungan produksi.</p>

## Menyimpan Model
<p>Untuk menyimpan model <span style="color:blue; background-color:gray; padding: 0.2% 0.5% 0.2% 0.5%;">GBR</span> yang telah dilatih, Anda dapat menggunakan modul joblib atau pickle pada Python. Kedua modul ini memungkinkan Anda untuk menyimpan model ke dalam sebuah file sehingga bisa digunakan kembali di masa mendatang tanpa perlu melatih ulang model. Mari kita bahas kedua cara tersebut secara saksama.</p>

##### 1. Joblib
Joblib adalah pilihan yang disarankan untuk menyimpan model scikit-learn karena lebih efisien dalam menyimpan objek model yang besar.

In [ ]:
import joblib
 
# Menyimpan model ke dalam file
joblib.dump(GBR, 'flask_deployment/gbr_model.joblib')

['gbr_model.joblib']

Tambahan: Digunakan untuk debug feature_names

In [93]:
import joblib

# Simpan nama - nama fitur
feature_names = X.columns.tolist()
joblib.dump(feature_names, 'flask_deployment/feature_names.pkl')

['flask_deployment/feature_names.pkl']

##### 2. Pickle
Pickle adalah modul standar Python yang dapat digunakan untuk menyimpan hampir semua objek Python termasuk model machine learning.

In [ ]:
import pickle
 
# Menyimpan model ke dalam file
with open('flask_deployment/gbr_model.pkl', 'wb') as file:
    pickle.dump(GBR, file)

<p>Kapan menggunakan joblib vs pickle? joblib lebih efisien dan cepat ketika bekerja dengan objek besar seperti model machine learning, sedangkan pickle lebih umum dan dapat digunakan untuk menyimpan berbagai jenis objek Python, tetapi kurang efisien dibandingkan joblib untuk model besar.</p>
<p>Memilih antara joblib dan pickle tergantung pada preferensi Anda, tetapi untuk menyimpan model scikit-learn, joblib sering kali merupakan pilihan yang lebih baik.</p>